# Sugar-Beets350×350：整图即单 patch → `.npy`

- **输入**：`img_350_crop` 或 `img_350_crop_ews_y` 下 **`rgb_*.png`**，每张 **350×350** RGB（一张图只对应 **一个** patch，不再切网格）。
- **输出**：`EWS/data/npy/`，每个源文件一个 **`float32` `.npy`**，形状 **`(350, 350, 3)`**，数值 \([0,1]\)（与预训练切块数组约定一致）。
- **命名**：`{源 stem}.npy`（例：`rgb_1461246987_538570251.npy`）。输出目录不存在时会自动创建。
- **进度**：若已安装 `tqdm` 则显示进度条。

In [1]:
from __future__ import annotations

from pathlib import Path

import numpy as np
from PIL import Image

PATCH = 350


def find_repo_with_ews(start: Path) -> Path:
    start = start.resolve()
    for p in [start, *start.parents]:
        if (p / "EWS" / "data").is_dir():
            return p
    raise FileNotFoundError("未找到 EWS/data，请在仓库根目录启动内核。")


REPO = find_repo_with_ews(Path.cwd())
OUT_DIR = REPO / "EWS" / "data" / "npy"

# 默认：亮度已对齐 EWS 的 350 图；可改为 REPO / "EWS" / "data" / "sugar-beets-2016-DatasetNinja" / "ds" / "img_350_crop"
IMG_DIR = (
    REPO
    / "EWS"
    / "data"
    / "sugar-beets-2016-DatasetNinja"
    / "ds"
    / "img_350_crop_ews_y"
)

# 若希望固定绝对路径：
# OUT_DIR = Path(r"D:\Cursor\UNSW-COMP-9517\EWS\data\npy")
# IMG_DIR = Path(r"D:\Cursor\UNSW-COMP-9517\EWS\data\sugar-beets-2016-DatasetNinja\ds\img_350_crop_ews_y")

OUT_DIR.mkdir(parents=True, exist_ok=True)

print("REPO:", REPO)
print("IMG_DIR:", IMG_DIR)
print("OUT_DIR:", OUT_DIR)

REPO: D:\Cursor\UNSW-COMP-9517
IMG_DIR: D:\Cursor\UNSW-COMP-9517\EWS\data\sugar-beets-2016-DatasetNinja\ds\img_350_crop_ews_y
OUT_DIR: D:\Cursor\UNSW-COMP-9517\EWS\data\npy


In [2]:
def png_to_patch_npy(src: Path, dst: Path) -> None:
    im = Image.open(src).convert("RGB")
    if im.size != (PATCH, PATCH):
        raise ValueError(f"期望 {PATCH}×{PATCH}，得到 {im.size[0]}×{im.size[1]}: {src.name}")
    arr = np.asarray(im, dtype=np.uint8)
    x = arr.astype(np.float32) / 255.0
    np.save(dst, x, allow_pickle=False)


if not IMG_DIR.is_dir():
    raise FileNotFoundError(f"输入目录不存在: {IMG_DIR}")

paths = sorted(IMG_DIR.glob("rgb_*.png"))
if not paths:
    raise FileNotFoundError(f"未找到 rgb_*.png: {IMG_DIR}")

print(f"待导出: {len(paths)} 张（每张1 个 .npy）")

待导出: 12714 张（每张1 个 .npy）


In [3]:
try:
    from tqdm import tqdm
except ImportError:
    tqdm = None

it = tqdm(paths, desc="→ npy") if tqdm else paths
for p in it:
    out_path = OUT_DIR / f"{p.stem}.npy"
    png_to_patch_npy(p, out_path)

print(f"完成: {len(paths)} 个 .npy → {OUT_DIR}")

→ npy: 100%|██████████| 12714/12714 [02:34<00:00, 82.09it/s] 

完成: 12714 个 .npy → D:\Cursor\UNSW-COMP-9517\EWS\data\npy


In [4]:
# 抽查第一个输出
chk = OUT_DIR / f"{paths[0].stem}.npy"
a = np.load(chk, mmap_mode=None)
print(chk.name, a.shape, a.dtype, float(a.min()), float(a.max()))

rgb_1461246987_538570251.npy (350, 350, 3) float32 0.11372549086809158 0.9921568632125854
